Data sources: Statistics Canada: combined [database_1](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1310011401) & [database_2](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1310014001) <i>("download entire table")</i><br>
[List of Canadian provinces and territories by life expectancy](https://en.wikipedia.org/wiki/List_of_Canadian_provinces_and_territories_by_life_expectancy) / [Продолжительность жизни в провинциях и территориях Канады](https://ru.wikipedia.org/wiki/Продолжительность_жизни_в_провинциях_и_территориях_Канады)<br>
[maps at Wikipedia Commons](https://commons.wikimedia.org/wiki/User:Lady3mlnm#Maps_for_Canada)

[MapChart](https://www.mapchart.net/canada.html)

In [2]:
import pandas as pd
import math
import re
from collections import namedtuple
import json

import sys
sys.path.append("..")
import mal_moduls_private.mal_total as mal

In [3]:
# load data
df_1 = pd.read_csv('data/Canada-3years-provinces-2024.zip', compression='zip',
                   usecols = ['REF_DATE', 'GEO', 'Age group', 'Sex', 'Element', 'VALUE'])

df_1 = df_1.loc[(df_1['Age group'] == '0 years') & (df_1['Element'] == 'Life expectancy (in years) at age x (ex)') & (df_1['REF_DATE'] >= '2009/2011')]

df_1.head(10)

,REF_DATE,GEO,Age group,Sex,Element,VALUE
869137,2009/2011,Canada,0 years,Both sexes,Life expectancy (in years) at age x (ex),81.40
869146,2009/2011,Canada,0 years,Males,Life expectancy (in years) at age x (ex),79.17
869155,2009/2011,Canada,0 years,Females,Life expectancy (in years) at age x (ex),83.52
872134,2009/2011,Newfoundland and Labrador,0 years,Both sexes,Life expectancy (in years) at age x (ex),79.60
872143,2009/2011,Newfoundland and Labrador,0 years,Males,Life expectancy (in years) at age x (ex),77.20
872152,2009/2011,Newfoundland and Labrador,0 years,Females,Life expectancy (in years) at age x (ex),82.08
875131,2009/2011,Nova Scotia,0 years,Both sexes,Life expectancy (in years) at age x (ex),80.25
875140,2009/2011,Nova Scotia,0 years,Males,Life expectancy (in years) at age x (ex),77.88
875149,2009/2011,Nova Scotia,0 years,Females,Life expectancy (in years) at age x (ex),82.51
878128,2009/2011,New Brunswick,0 years,Both sexes,Life expectancy (in years) at age x (ex),80.66


In [4]:
df_1 = df_1.drop(columns=['Age group', 'Element']) \
           .rename(columns={'REF_DATE': 'year',
                            'GEO': 'region',
                            'Sex': 'sex',
                            'VALUE': 'value'})
df_1

,year,region,sex,value
869137,2009/2011,Canada,Both sexes,81.40
869146,2009/2011,Canada,Males,79.17
869155,2009/2011,Canada,Females,83.52
872134,2009/2011,Newfoundland and Labrador,Both sexes,79.60
872143,2009/2011,Newfoundland and Labrador,Males,77.20
...,...,...,...,...
1282732,2022/2024,Alberta,Males,78.52
1282741,2022/2024,Alberta,Females,83.20
1285720,2022/2024,British Columbia,Both sexes,81.98
1285729,2022/2024,British Columbia,Males,79.38


<br>
<br>

In [6]:
# load data
df_2 = pd.read_csv('data/Canada-3years-small_terrotories-2024.zip', compression='zip',
                   usecols = ['REF_DATE', 'GEO', 'Age interval', 'Sex', 'Element', 'VALUE'])

df_2 = df_2.loc[(df_2['Age interval'] == '0 years') & (df_2['Element'] == 'Life expectancy (in years) from age interval i (ei)') & (df_2['REF_DATE'] >= '2009/2011')]

df_2.head(10)

,REF_DATE,GEO,Age interval,Sex,Element,VALUE
62647,2009/2011,Prince Edward Island,0 years,Both sexes,Life expectancy (in years) from age interval i...,80.56
62656,2009/2011,Prince Edward Island,0 years,Males,Life expectancy (in years) from age interval i...,77.94
62665,2009/2011,Prince Edward Island,0 years,Females,Life expectancy (in years) from age interval i...,83.01
63187,2009/2011,Yukon,0 years,Both sexes,Life expectancy (in years) from age interval i...,77.99
63196,2009/2011,Yukon,0 years,Males,Life expectancy (in years) from age interval i...,75.68
63205,2009/2011,Yukon,0 years,Females,Life expectancy (in years) from age interval i...,80.31
63727,2009/2011,Northwest Territories,0 years,Both sexes,Life expectancy (in years) from age interval i...,77.59
63736,2009/2011,Northwest Territories,0 years,Males,Life expectancy (in years) from age interval i...,75.53
63745,2009/2011,Northwest Territories,0 years,Females,Life expectancy (in years) from age interval i...,80.22
64267,2009/2011,Nunavut,0 years,Both sexes,Life expectancy (in years) from age interval i...,71.25


In [7]:
df_2 = df_2.drop(columns=['Age interval', 'Element']) \
           .rename(columns={'REF_DATE': 'year',
                            'GEO': 'region',
                            'Sex': 'sex',
                            'VALUE': 'value'})
df_2

,year,region,sex,value
62647,2009/2011,Prince Edward Island,Both sexes,80.56
62656,2009/2011,Prince Edward Island,Males,77.94
62665,2009/2011,Prince Edward Island,Females,83.01
63187,2009/2011,Yukon,Both sexes,77.99
63196,2009/2011,Yukon,Males,75.68
...,...,...,...,...
91816,2022/2024,Northwest Territories,Males,73.86
91825,2022/2024,Northwest Territories,Females,79.11
92347,2022/2024,Nunavut,Both sexes,70.84
92356,2022/2024,Nunavut,Males,69.50


<br>
<br>

In [9]:
# combine data from two database
df = pd.concat([df_1, df_2])

del df_1, df_2

df

,year,region,sex,value
869137,2009/2011,Canada,Both sexes,81.40
869146,2009/2011,Canada,Males,79.17
869155,2009/2011,Canada,Females,83.52
872134,2009/2011,Newfoundland and Labrador,Both sexes,79.60
872143,2009/2011,Newfoundland and Labrador,Males,77.20
...,...,...,...,...
91816,2022/2024,Northwest Territories,Males,73.86
91825,2022/2024,Northwest Territories,Females,79.11
92347,2022/2024,Nunavut,Both sexes,70.84
92356,2022/2024,Nunavut,Males,69.50


In [10]:
df_total = df.loc[df['sex'] == 'Both sexes'] \
             .pivot(index='region', columns='year', values='value')

df_total.index.name = df_total.columns.name = ''

print(df_total.shape)
df_total.fillna('')

(14, 14)


,2009/2011,2010/2012,2011/2013,2012/2014,2013/2015,2014/2016,2015/2017,2016/2018,2017/2019,2018/2020,2019/2021,2020/2022,2021/2023,2022/2024
,,,,,,,,,,,,,,
Alberta,81.14,81.33,81.37,81.37,81.39,81.44,81.41,81.43,81.58,81.39,80.98,80.4,80.36,80.81
British Columbia,82.15,82.36,82.47,82.59,82.61,82.54,82.33,82.19,82.32,82.32,82.03,81.6,81.57,81.98
Canada,81.40,81.59,81.73,81.82,81.88,81.94,81.92,81.91,81.99,81.88,81.76,81.39,81.43,81.65
Manitoba,79.77,79.95,80.07,80.10,80.10,79.99,79.98,79.96,80.03,79.8,79.55,78.96,78.89,78.9
New Brunswick,80.66,80.86,80.97,80.96,80.93,80.79,80.67,80.64,80.69,80.78,80.83,80.44,80.25,80.12
Newfoundland and Labrador,79.60,79.60,79.52,79.37,79.27,79.39,79.57,79.83,79.97,79.96,79.73,79.33,79.14,79.17
Northwest Territories,77.59,77.91,77.82,77.65,77.65,77.22,77.13,76.97,77.59,77.49,77.38,76.1,76.63,76.29
Nova Scotia,80.25,80.34,80.37,80.48,80.36,80.44,80.41,80.48,80.4,80.46,80.48,80.25,80.21,80.3
Nunavut,71.25,71.65,70.21,70.73,71.00,71.66,71.82,71.19,70.65,70.57,70.69,71.21,70.78,70.84


In [11]:
# just for interest, explore results: determine regions with max and min values, and also look at specific regions
mal.min_and_max_values(df_total, max_lng=11, row_center='Canada')

Number of records: 14


,2009/2011,2010/2012,2011/2013,2012/2014,2013/2015,2014/2016,2015/2017,2016/2018,2017/2019,2018/2020,2019/2021,2020/2022,2021/2023,2022/2024
max,82.15 -British Co…,82.36 -British Co…,82.47 -British Co…,82.59 -British Co…,82.61 -British Co…,82.54 -British Co…,82.41 -Ontario,82.44 -Quebec,82.56 -Quebec,82.46 -Quebec,82.6 -Quebec,82.32 -Quebec,82.44 -Quebec,82.39 -Quebec
max_2,81.78 -Ontario,82.01 -Ontario,82.19 -Ontario,82.29 -Ontario,82.36 -Ontario,82.41 -Ontario,82.34 -Quebec,82.35 -Ontario,82.39 -Ontario,82.32 -British Co…,82.14 -Ontario,81.86 -Ontario,81.96 -Ontario,82.27 -Ontario
max_3,81.46 -Quebec,81.6 -Quebec,81.77 -Quebec,81.92 -Quebec,82.05 -Quebec,82.24 -Quebec,82.33 -British Co…,82.19 -British Co…,82.32 -British Co…,82.27 -Ontario,82.03 -British Co…,81.6 -British Co…,81.57 -British Co…,81.98 -British Co…
Canada,– 81.4 –,– 81.59 –,– 81.73 –,– 81.82 –,– 81.88 –,– 81.94 –,– 81.92 –,– 81.91 –,– 81.99 –,– 81.88 –,– 81.76 –,– 81.39 –,– 81.43 –,– 81.65 –
min_3,77.99 -Yukon,78.46 -Yukon,78.61 -Yukon,78.53 -Yukon,78.56 -Yukon,78.67 -Yukon,79.57 -Newfoundla…,79.83 -Newfoundla…,79.97 -Newfoundla…,79.8 -Manitoba,79.41 -Saskatchew…,78.76 -Saskatchew…,78.65 -Saskatchew…,78.9 -Manitoba
min_2,77.59 -Northwest …,77.91 -Northwest …,77.82 -Northwest …,77.65 -Northwest …,77.65 -Northwest …,77.22 -Northwest …,77.13 -Northwest …,76.97 -Northwest …,77.59 -Northwest …,77.49 -Northwest …,77.38 -Northwest …,76.1 -Northwest …,76.63 -Northwest …,76.29 -Northwest …
min,71.25 -Nunavut,71.65 -Nunavut,70.21 -Nunavut,70.73 -Nunavut,71.0 -Nunavut,71.66 -Nunavut,71.82 -Nunavut,71.19 -Nunavut,70.65 -Nunavut,70.57 -Nunavut,70.69 -Nunavut,71.21 -Nunavut,70.78 -Nunavut,70.84 -Nunavut


In [12]:
df_male = df.loc[df['sex'] == 'Males'] \
             .pivot(index='region', columns='year', values='value')

df_male.index.name = df_male.columns.name = ''

print(df_male.shape)
df_male.fillna('')

(14, 14)


,2009/2011,2010/2012,2011/2013,2012/2014,2013/2015,2014/2016,2015/2017,2016/2018,2017/2019,2018/2020,2019/2021,2020/2022,2021/2023,2022/2024
,,,,,,,,,,,,,,
Alberta,78.93,79.12,79.22,79.28,79.28,79.28,79.17,79.15,79.33,79.1,78.58,77.95,77.93,78.52
British Columbia,80.07,80.31,80.42,80.52,80.52,80.40,80.08,79.82,79.95,79.87,79.48,78.89,78.87,79.38
Canada,79.17,79.40,79.57,79.70,79.79,79.85,79.82,79.79,79.86,79.72,79.5,79.1,79.16,79.47
Manitoba,77.54,77.76,77.83,77.85,77.91,77.84,77.87,77.82,77.87,77.61,77.21,76.49,76.43,76.49
New Brunswick,78.24,78.45,78.63,78.76,78.77,78.67,78.5,78.57,78.52,78.65,78.69,78.35,78.19,78.07
Newfoundland and Labrador,77.20,77.24,77.32,77.14,77.25,77.21,77.49,77.82,77.97,77.93,77.69,77.33,77.05,77.14
Northwest Territories,75.53,75.45,75.87,76.18,76.08,75.57,75.24,75.16,75.66,75.28,75.02,73.36,73.72,73.86
Nova Scotia,77.88,78.08,78.12,78.23,78.14,78.19,78.2,78.26,78.34,78.4,78.35,78.06,78.0,78.13
Nunavut,69.02,69.38,68.44,68.28,69.03,70.01,70.33,69.35,68.6,67.83,68.05,68.47,68.5,69.5


In [13]:
df_female = df.loc[df['sex'] == 'Females'] \
             .pivot(index='region', columns='year', values='value')

df_female.index.name = df_female.columns.name = ''

print(df_female.shape)
df_female.fillna('')

(14, 14)


,2009/2011,2010/2012,2011/2013,2012/2014,2013/2015,2014/2016,2015/2017,2016/2018,2017/2019,2018/2020,2019/2021,2020/2022,2021/2023,2022/2024
,,,,,,,,,,,,,,
Alberta,83.37,83.56,83.53,83.49,83.52,83.64,83.72,83.79,83.91,83.76,83.48,82.99,82.92,83.2
British Columbia,84.20,84.39,84.52,84.67,84.70,84.67,84.59,84.61,84.75,84.85,84.66,84.44,84.39,84.69
Canada,83.52,83.68,83.78,83.86,83.88,83.95,83.97,83.99,84.09,84.03,84.01,83.71,83.72,83.86
Manitoba,81.99,82.12,82.32,82.37,82.31,82.16,82.1,82.14,82.23,82.07,81.97,81.55,81.49,81.45
New Brunswick,83.02,83.24,83.28,83.12,83.07,82.94,82.89,82.71,82.8,82.82,82.99,82.56,82.38,82.24
Newfoundland and Labrador,82.08,82.10,81.71,81.69,81.40,81.62,81.65,81.92,81.98,81.98,81.87,81.42,81.21,81.22
Northwest Territories,80.22,80.13,79.83,79.21,78.96,79.00,79.23,78.72,79.64,79.9,80.27,79.15,79.75,79.11
Nova Scotia,82.51,82.50,82.51,82.70,82.57,82.68,82.55,82.63,82.43,82.53,82.62,82.46,82.51,82.53
Nunavut,74.13,74.36,72.84,73.83,73.20,73.22,73.41,72.94,72.76,73.06,73.82,74.99,73.27,72.17


<br>
<br>

In [15]:
df_provinces = pd.concat([
        df_total['2017/2019'], df_male['2017/2019'], df_female['2017/2019'], (df_female['2017/2019']-df_male['2017/2019']).round(2),
        (df_total['2021/2023']-df_total['2017/2019']).round(2),
        df_total['2021/2023'], df_male['2021/2023'], df_female['2021/2023'], (df_female['2021/2023']-df_male['2021/2023']).round(2),
        (df_total['2022/2024']-df_total['2021/2023']).round(2),
        df_total['2022/2024'], df_male['2022/2024'], df_female['2022/2024'], (df_female['2022/2024']-df_male['2022/2024']).round(2),
        (df_total['2022/2024']-df_total['2017/2019']).round(2)],
    axis='columns',
    keys=['2017-19_t', '2017-19_m', '2017-19_f', '2017-19_fΔm', 'Δ1',
          '2021-23_t', '2021-23_m', '2021-23_f', '2021-23_fΔm', 'Δ2',
          '2022-24_t', '2022-24_m', '2022-24_f', '2022-24_fΔm', 'Δtotal']
)

del df, df_total, df_male, df_female

df_provinces.dropna(how='all', inplace=True)

df_provinces.sort_values(['2022-24_t', '2022-24_m', '2022-24_f'], ascending=False, inplace=True)
df_provinces = pd.concat([df_provinces.loc[['Canada']], df_provinces.drop(index='Canada')])

df_provinces.fillna('')

,2017-19_t,2017-19_m,2017-19_f,2017-19_fΔm,Δ1,2021-23_t,2021-23_m,2021-23_f,2021-23_fΔm,Δ2,2022-24_t,2022-24_m,2022-24_f,2022-24_fΔm,Δtotal
,,,,,,,,,,,,,,,
Canada,81.99,79.86,84.09,4.23,-0.56,81.43,79.16,83.72,4.56,0.22,81.65,79.47,83.86,4.39,-0.34
Quebec,82.56,80.68,84.34,3.66,-0.12,82.44,80.56,84.26,3.70,-0.05,82.39,80.60,84.14,3.54,-0.17
Ontario,82.39,80.25,84.46,4.21,-0.43,81.96,79.68,84.23,4.55,0.31,82.27,80.06,84.47,4.41,-0.12
British Columbia,82.32,79.95,84.75,4.80,-0.75,81.57,78.87,84.39,5.52,0.41,81.98,79.38,84.69,5.31,-0.34
Prince Edward Island,81.21,78.96,83.41,4.45,-0.32,80.89,78.58,83.11,4.53,0.27,81.16,79.01,83.20,4.19,-0.05
Alberta,81.58,79.33,83.91,4.58,-1.22,80.36,77.93,82.92,4.99,0.45,80.81,78.52,83.20,4.68,-0.77
Nova Scotia,80.40,78.34,82.43,4.09,-0.19,80.21,78.00,82.51,4.51,0.09,80.30,78.13,82.53,4.40,-0.10
New Brunswick,80.69,78.52,82.80,4.28,-0.44,80.25,78.19,82.38,4.19,-0.13,80.12,78.07,82.24,4.17,-0.57
Newfoundland and Labrador,79.97,77.97,81.98,4.01,-0.83,79.14,77.05,81.21,4.16,0.03,79.17,77.14,81.22,4.08,-0.80


In [16]:
mal.min_and_max_values(df_provinces[['2017-19_t', 'Δ1', '2021-23_t', 'Δ2', '2022-24_t', 'Δtotal']], max_lng=21, row_center='Canada')

Number of records: 13


,2017-19_t,Δ1,2021-23_t,Δ2,2022-24_t,Δtotal
max,82.56 -Quebec,0.13 -Nunavut,82.44 -Quebec,0.45 -Alberta,82.39 -Quebec,0.19 -Nunavut
max_2,82.39 -Ontario,-0.12 -Quebec,81.96 -Ontario,0.41 -British Columbia,82.27 -Ontario,-0.05 -Prince Edward Island
max_3,82.32 -British Columbia,-0.19 -Nova Scotia,81.57 -British Columbia,0.37 -Saskatchewan,81.98 -British Columbia,-0.1 -Nova Scotia
Canada,– 81.99 –,– -0.56 –,– 81.43 –,– 0.22 –,– 81.65 –,– -0.34 –
min_3,79.97 -Newfoundland and Lab…,-1.14 -Manitoba,78.65 -Saskatchewan,-0.05 -Quebec,78.9 -Manitoba,-1.13 -Manitoba
min_2,77.59 -Northwest Territories,-1.22 -Alberta,76.63 -Northwest Territories,-0.13 -New Brunswick,76.29 -Northwest Territories,-1.25 -Saskatchewan
min,70.65 -Nunavut,-1.62 -Saskatchewan,70.78 -Nunavut,-0.34 -Northwest Territories,70.84 -Nunavut,-1.3 -Northwest Territories


In [17]:
mal.min_and_max_values(df_provinces[['2017-19_m', '2021-23_m', '2022-24_m', '2017-19_f', '2021-23_f', '2022-24_f']], max_lng=21, row_center='Canada')

Number of records: 13


,2017-19_m,2021-23_m,2022-24_m,2017-19_f,2021-23_f,2022-24_f
max,80.68 -Quebec,80.56 -Quebec,80.6 -Quebec,84.75 -British Columbia,84.39 -British Columbia,84.69 -British Columbia
max_2,80.25 -Ontario,79.68 -Ontario,80.06 -Ontario,84.46 -Ontario,84.26 -Quebec,84.47 -Ontario
max_3,79.95 -British Columbia,79.16 -Canada,79.47 -Canada,84.34 -Quebec,84.23 -Ontario,84.14 -Quebec
Canada,– 79.86 –,– 79.16 –,– 79.47 –,– 84.09 –,– 83.72 –,– 83.86 –
min_3,77.87 -Manitoba,76.3 -Saskatchewan,76.49 -Manitoba,81.98 -Newfoundland and Lab…,81.17 -Saskatchewan,81.22 -Newfoundland and Lab…
min_2,75.66 -Northwest Territories,73.72 -Northwest Territories,73.86 -Northwest Territories,79.64 -Northwest Territories,79.75 -Northwest Territories,79.11 -Northwest Territories
min,68.6 -Nunavut,68.5 -Nunavut,69.5 -Nunavut,72.76 -Nunavut,73.27 -Nunavut,72.17 -Nunavut


<br>
<br>

In [19]:
# for region in sorted(df_provinces.index.to_list()):
#     print(f"    '{region}': {{'en': ('', ''), 'ru': ('', '')}},")

In [20]:
dd_replacement = {
    'Canada': {'en': ('Canada on average', ''), 'ru': ('Канада в среднем', '')},
    'Alberta': {'en': ('Alberta', 'Alberta'), 'ru': ('Альбе́рта', 'Альберта')},
    'British Columbia': {'en': ('British Columbia', 'British Columbia'), 'ru': ('Британская Колу́мбия', 'Британская Колумбия')},
    'Manitoba': {'en': ('Manitoba', 'Manitoba'), 'ru': ('Манито́ба', 'Манитоба')},
    'New Brunswick': {'en': ('New Brunswick', 'New Brunswick'), 'ru': ('Нью-Бра́нсуик', 'Нью-Брансуик')},
    'Newfoundland and Labrador': {'en': ('Newfoundland and Labrador', 'Newfoundland and Labrador'), 'ru': ('Ньюфа́ундленд и Лабрадо́р', 'Ньюфаундленд и Лабрадор')},
    'Northwest Territories': {'en': ('Northwest Territories', 'Northwest Territories'), 'ru': ('Северо-Западные территории', 'Северо-Западные территории')},
    'Nova Scotia': {'en': ('Nova Scotia', 'Nova Scotia'), 'ru': ('Новая Шотла́ндия', 'Новая Шотландия')},
    'Nunavut': {'en': ('Nunavut', 'Nunavut'), 'ru': ('Ну́навут', 'Нунавут')},
    'Ontario': {'en': ('Ontario', 'Ontario'), 'ru': ('Онта́рио', 'Онтарио')},
    'Prince Edward Island': {'en': ('Prince Edward Island', 'Prince Edward Island'), 'ru': ('Остров Принца Эдуарда', 'Остров Принца Эдуарда')},
    'Quebec': {'en': ('Quebec', 'Quebec'), 'ru': ('Квебе́к', 'Квебек')},
    'Saskatchewan': {'en': ('Saskatchewan', 'Saskatchewan'), 'ru': ('Саскaчевáн', 'Саскачеван')},
    'Yukon': {'en': ('Yukon', 'Yukon'), 'ru': ('Ю́кон', 'Юкон (территория)')},
}

In [21]:
# create code for placing info in Wikipedia
def create_table_regions(df, file_header, lang='en'):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)

    def chval(x, prec=2, *, add_par=''):  # change_value
        return f'style="padding-right:1.5ex;{add_par}"| —' if math.isnan(x) else \
               f'style="padding-right:1.5ex;color:darkgreen;{add_par}"| {x:0.{prec}f}' if x>0 else \
               f'style="padding-right:1.5ex;color:crimson;{add_par}"| −{-x:0.{prec}f}' if x<0 else \
               f'style="padding-right:1.5ex;color:darkgray;{add_par}"| {x:0.{prec}f}'
    
    def chval_bold(x, prec=2, *, add_par=''):  # change_value
        return ' \'\'\'—\'\'\'' if math.isnan(x) else \
               f'style="padding-right:1.5ex;color:darkgreen;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="padding-right:1.5ex;color:crimson;{add_par}"| \'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="padding-right:1.5ex;color:darkgray;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\''

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]    
        if ser.name == 'Canada':
             st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="text-align:center;background:#e0ffd8;"|\'\'\'{if_value(ser["2017-19_t"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#eaf3ff;"|\'\'\'{if_value(ser["2017-19_m"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#fee7f6;"|\'\'\'{if_value(ser["2017-19_f"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#fff8dc;"|\'\'\'{if_value(ser["2017-19_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["Δ1"], add_par="border-left-width:2px;")} ' + \
                  f'||style="text-align:center;background:#e0ffd8;border-left-width:2px;"|\'\'\'{if_value(ser["2021-23_t"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#eaf3ff;"|\'\'\'{if_value(ser["2021-23_m"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#fee7f6;"|\'\'\'{if_value(ser["2021-23_f"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#fff8dc;"|\'\'\'{if_value(ser["2021-23_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["Δ2"], add_par="border-left-width:2px;")} ' + \
                  f'||style="text-align:center;background:#e0ffd8;border-left-width:2px;"|\'\'\'{if_value(ser["2022-24_t"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#eaf3ff;"|\'\'\'{if_value(ser["2022-24_m"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#fee7f6;"|\'\'\'{if_value(ser["2022-24_f"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#fff8dc;"|\'\'\'{if_value(ser["2022-24_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["Δtotal"], add_par="border-left-width:2px;")}'
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="text-align:center;background:#e0ffd8;"|\'\'\'{if_value(ser["2017-19_t"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#eaf3ff;"|{if_value(ser["2017-19_m"])} ' + \
                  f'||style="text-align:center;background:#fee7f6;"|{if_value(ser["2017-19_f"])} ' + \
                  f'||style="text-align:center;background:#fff8dc;"|{if_value(ser["2017-19_fΔm"])} ' + \
                  f'||{chval(ser["Δ1"], add_par="border-left-width:2px;")} ' + \
                  f'||style="text-align:center;background:#e0ffd8;border-left-width:2px;"|\'\'\'{if_value(ser["2021-23_t"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#eaf3ff;"|{if_value(ser["2021-23_m"])} ' + \
                  f'||style="text-align:center;background:#fee7f6;"|{if_value(ser["2021-23_f"])} ' + \
                  f'||style="text-align:center;background:#fff8dc;"|{if_value(ser["2021-23_fΔm"])} ' + \
                  f'||{chval(ser["Δ2"], add_par="border-left-width:2px;")} ' + \
                  f'||style="text-align:center;background:#e0ffd8;border-left-width:2px;"|\'\'\'{if_value(ser["2022-24_t"])}\'\'\' ' + \
                  f'||style="text-align:center;background:#eaf3ff;"|{if_value(ser["2022-24_m"])} ' + \
                  f'||style="text-align:center;background:#fee7f6;"|{if_value(ser["2022-24_f"])} ' + \
                  f'||style="text-align:center;background:#fff8dc;"|{if_value(ser["2022-24_fΔm"])} ' + \
                  f'||{chval(ser["Δtotal"], add_par="border-left-width:2px;")}'
        
    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —')

    return st


table_code = create_table_regions(df_provinces, file_header='Canada_header_3years_en -2024.txt', lang='en')

# write the code to file
with open('output/Table code for Canada -3_years -en.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [22]:
table_code = create_table_regions(df_provinces, file_header='Canada_header_3years_ru -2024.txt', lang='ru')

# write the code to file
with open('output/Table code for Canada -3_years -ru.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

<br>
<br>
<br>
<hr>

<h3>Map creation</h3>

In [24]:
CountryGroup = namedtuple('CountryGroup', ['group_label', 'color', 'countries'])

In [25]:
# for state in sorted(df_provinces.index.to_list()):
#     print(f"               '{state}' : ''")

In [26]:
df_map = df_provinces.copy()                                    \
                     .drop(['Canada']) \
                     .rename(index={ })

df_map.head(3).fillna('')

,2017-19_t,2017-19_m,2017-19_f,2017-19_fΔm,Δ1,2021-23_t,2021-23_m,2021-23_f,2021-23_fΔm,Δ2,2022-24_t,2022-24_m,2022-24_f,2022-24_fΔm,Δtotal
,,,,,,,,,,,,,,,
Quebec,82.56,80.68,84.34,3.66,-0.12,82.44,80.56,84.26,3.70,-0.05,82.39,80.60,84.14,3.54,-0.17
Ontario,82.39,80.25,84.46,4.21,-0.43,81.96,79.68,84.23,4.55,0.31,82.27,80.06,84.47,4.41,-0.12
British Columbia,82.32,79.95,84.75,4.80,-0.75,81.57,78.87,84.39,5.52,0.41,81.98,79.38,84.69,5.31,-0.34


In [27]:
dd_legend = {
    '82.5–82.99' : '004000',
    '82.0–82.49' : '006000',
    '81.5–81.99' : '009000',
    '81.0–81.49' : '00b800',
    '80.5–80.99' : '00e000',
    '80.0–80.49' : '00ff00',
    '79.5–79.99' : 'b8ff00',
    '79.0–79.49' : 'ffff00',
    '78.5–78.99' : 'ffe000',
    '78.0–78.49' : 'ffc000',
    '77.5–77.99' : 'ffa000',
    '77.0–77.49' : 'ff8000',
    '76.5–76.99' : 'ff5000',
    '76.0–76.49' : 'ff0000',
    '75.5–75.99' : 'c80000',
    '75.0–75.49' : '900000',
    '70.0–74.99' : '400000'
}

def create_legend_code(dd_legend):
    for k, v in dd_legend.items():
        print(f"{{{{Legend|#{v}|{k}}}}}")

create_legend_code(dd_legend)

{{Legend|#004000|82.5–82.99}}
{{Legend|#006000|82.0–82.49}}
{{Legend|#009000|81.5–81.99}}
{{Legend|#00b800|81.0–81.49}}
{{Legend|#00e000|80.5–80.99}}
{{Legend|#00ff00|80.0–80.49}}
{{Legend|#b8ff00|79.5–79.99}}
{{Legend|#ffff00|79.0–79.49}}
{{Legend|#ffe000|78.5–78.99}}
{{Legend|#ffc000|78.0–78.49}}
{{Legend|#ffa000|77.5–77.99}}
{{Legend|#ff8000|77.0–77.49}}
{{Legend|#ff5000|76.5–76.99}}
{{Legend|#ff0000|76.0–76.49}}
{{Legend|#c80000|75.5–75.99}}
{{Legend|#900000|75.0–75.49}}
{{Legend|#400000|70.0–74.99}}


In [28]:
SELECTED_COL = '2022-24_t'
df_grouped = mal.bin_values_in_dataframe(df_map, SELECTED_COL, step = 0.5)

df_grouped['group_label'] = df_grouped['group_label'].map(lambda st: '70.0–74.99' if st < '75.0–75.49' else st)

df_grouped

Range: 70.84 – 82.39   (Nunavut – Quebec)
Number of groups: 9
Number of values: 12


,2022-24_t,group_label
,,
Quebec,82.39,82.0–82.49
Ontario,82.27,82.0–82.49
British Columbia,81.98,81.5–81.99
Prince Edward Island,81.16,81.0–81.49
Alberta,80.81,80.5–80.99
Nova Scotia,80.30,80.0–80.49
New Brunswick,80.12,80.0–80.49
Newfoundland and Labrador,79.17,79.0–79.49
Saskatchewan,79.02,79.0–79.49


In [29]:
def extract_indexes(subdf, dd_legend = dd_legend):
    group_label = subdf['group_label'].iloc[0]
    countries = subdf.index.to_list()
    color = (dd_legend[group_label])
    
    ls_grouping.append(CountryGroup(group_label=group_label, countries=countries, color=color))

    return pd.Series([color, countries], index=['color', 'regions'])


ls_grouping = []
df_grouped = df_grouped.groupby(['group_label'])[['group_label']].apply(extract_indexes).loc[::-1]

df_grouped

,color,regions
group_label,,
82.0–82.49,006000,"[Quebec, Ontario]"
81.5–81.99,009000,[British Columbia]
81.0–81.49,00b800,[Prince Edward Island]
80.5–80.99,00e000,[Alberta]
80.0–80.49,00ff00,"[Nova Scotia, New Brunswick]"
79.0–79.49,ffff00,"[Newfoundland and Labrador, Saskatchewan]"
78.5–78.99,ffe000,[Manitoba]
76.0–76.49,ff0000,[Northwest Territories]
70.0–74.99,400000,[Nunavut]


In [30]:
if ls_grouping[0].group_label == '70.0–74.99':
    ls_grouping[0] = ls_grouping[0]._replace(group_label = '<75')

if 'Yukon' not in df_grouped.index.to_list():
    ls_grouping.insert(0, CountryGroup(group_label='n/a', countries=['Yukon'], color='e0e0e0'))

ls_grouping

[CountryGroup(group_label='n/a', color='e0e0e0', countries=['Yukon']),
 CountryGroup(group_label='<75', color='400000', countries=['Nunavut']),
 CountryGroup(group_label='76.0–76.49', color='ff0000', countries=['Northwest Territories']),
 CountryGroup(group_label='78.5–78.99', color='ffe000', countries=['Manitoba']),
 CountryGroup(group_label='79.0–79.49', color='ffff00', countries=['Newfoundland and Labrador', 'Saskatchewan']),
 CountryGroup(group_label='80.0–80.49', color='00ff00', countries=['Nova Scotia', 'New Brunswick']),
 CountryGroup(group_label='80.5–80.99', color='00e000', countries=['Alberta']),
 CountryGroup(group_label='81.0–81.49', color='00b800', countries=['Prince Edward Island']),
 CountryGroup(group_label='81.5–81.99', color='009000', countries=['British Columbia']),
 CountryGroup(group_label='82.0–82.49', color='006000', countries=['Quebec', 'Ontario'])]

In [31]:
def create_json(ls_grouping, title=''):
    jo = {
          "groups": { },
          "title": title,
          "hidden": [],
          "background": "#fff",
          "borders": "#000",
          "legendFont": "Century Gothic",
          "legendFontColor": "#000",
          "legendBgColor": "#00000000",
          "legendBoxShape": "square",
          "legendBorderColor": "#00000000",
          "legendWidth": 287,
          "areBordersShown": True,
          "defaultColor": "#d1dbdd",
          "labelsColor": "#000000",
          "labelsFont": "Arial",
          "strokeWidth": "medium",
          "areLabelsShown": True,
          "uncoloredScriptColor": "#ffff33",
          "v5": True,
          "legendPosition": "custom",
          "legendX": 1350,
          "legendY": 120,
          "canvasWidth": 1722,
          "canvasHeight": 1490,
          "legendSize": "large",
          "legendStatus": "show",
          "scalingPatterns": True,
          "legendRowsSameColor": True,
          "legendColumnCount": 1
        }
    
    for group_label, color, regions in ls_grouping[::-1]:
        jo["groups"][f"#{color}"] = {"label": group_label,
                                     "paths": [region.replace(' ', '_').replace('Labrador', 'Labrador_') for region in regions]}       

    return jo


jo = create_json(ls_grouping, title=SELECTED_COL[:-2].replace('-', ' – 20'))

pretty_jo = json.dumps(jo, indent=2)

with open(f"output/map_JSON -{SELECTED_COL[:-2]}.txt", 'w', encoding="utf-8") as fh:
    fh.write(pretty_jo)